# AI Research Assistant Agent (Agentic AI)

**Stack:** Python, LangChain-style orchestration, MCP-style tool protocol, RAG, LLM APIs (OpenAI/Anthropic)

An autonomous multi-agent system that:
1. **Plans** — breaks a complex query into sub-tasks
2. **Retrieves** — grounds answers in real documents via a RAG pipeline (vector embeddings + retrieval)
3. **Acts** — dynamically selects and calls external tools through an MCP-style tool protocol
4. **Synthesizes** — combines retrieved evidence + tool outputs into a final answer, with memory of intermediate steps

**LLM note:** This notebook uses a pluggable `LLMClient` — it calls a real OpenAI/Anthropic model if an API
key is present in the environment, and otherwise falls back to a deterministic local planner so the full
pipeline runs end-to-end with zero errors and no API key required. Swap in your key to see it run against
a real LLM with no other code changes.


In [1]:
import os
import re
import json
from datetime import date
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


## 1. Knowledge base + RAG retrieval

In [2]:
DOCUMENTS = [
    "Refund Policy: Customers can request a full refund within 30 days of purchase. "
    "Refunds are processed to the original payment method within 5-7 business days.",

    "Shipping Policy: Standard shipping takes 5-8 business days. Express shipping takes 2-3 business days "
    "and costs an additional flat fee of $12.",

    "Product Warranty: All electronics come with a 1-year manufacturer warranty covering defects. "
    "Accidental damage is not covered under the standard warranty.",

    "Support Hours: Customer support is available Monday to Saturday, 9 AM to 9 PM IST. "
    "Support is unavailable on national holidays.",

    "Loyalty Program: Members earn 2 points per dollar spent. 100 points can be redeemed for a $5 discount "
    "on any future order.",
]

class RAGRetriever:
    """Vector-embedding based retriever grounding agent responses in real documents."""
    def __init__(self, documents):
        self.documents = documents
        self.vectorizer = TfidfVectorizer().fit(documents)
        self.doc_vectors = self.vectorizer.transform(documents)

    def retrieve(self, query, k=2):
        q_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(q_vec, self.doc_vectors)[0]
        top_idx = np.argsort(sims)[::-1][:k]
        return [(self.documents[i], float(sims[i])) for i in top_idx if sims[i] > 0]

retriever = RAGRetriever(DOCUMENTS)
for doc, score in retriever.retrieve("How long do refunds take?"):
    print(f"[{score:.3f}] {doc}")


[0.198] Refund Policy: Customers can request a full refund within 30 days of purchase. Refunds are processed to the original payment method within 5-7 business days.


## 2. MCP-style tool protocol

Tools are registered with a name, description, and callable — the same contract the Model Context
Protocol uses to let an agent discover and dynamically invoke external tools/APIs at runtime.

In [3]:
class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, name, description):
        def decorator(fn):
            self._tools[name] = {"fn": fn, "description": description}
            return fn
        return decorator

    def list_tools(self):
        return {name: t["description"] for name, t in self._tools.items()}

    def call(self, name, **kwargs):
        if name not in self._tools:
            raise ValueError(f"Unknown tool: {name}")
        return self._tools[name]["fn"](**kwargs)

tools = ToolRegistry()

@tools.register("search_documents", "Retrieve relevant company documents for a query")
def search_documents_tool(query: str):
    results = retriever.retrieve(query, k=2)
    return [doc for doc, score in results]

@tools.register("calculator", "Evaluate a simple arithmetic expression")
def calculator_tool(expression: str):
    safe_expr = re.sub(r"[^0-9+\-*/(). ]", "", expression)
    return eval(safe_expr, {"__builtins__": {}})

@tools.register("get_current_date", "Get today\'s date")
def get_date_tool():
    return str(date.today())

print("Registered MCP-style tools:")
for name, desc in tools.list_tools().items():
    print(f"- {name}: {desc}")


Registered MCP-style tools:
- search_documents: Retrieve relevant company documents for a query
- calculator: Evaluate a simple arithmetic expression
- get_current_date: Get today's date


## 3. Pluggable LLM client (OpenAI / Anthropic / local fallback)

Tries a real LLM API if a key is present in the environment; otherwise falls back to a deterministic
local responder so the pipeline is fully runnable without external dependencies.

In [4]:
class LLMClient:
    def __init__(self):
        self.backend = "local"
        if os.environ.get("ANTHROPIC_API_KEY"):
            self.backend = "anthropic"
        elif os.environ.get("OPENAI_API_KEY"):
            self.backend = "openai"

    def generate(self, prompt: str) -> str:
        if self.backend == "anthropic":
            import anthropic
            client = anthropic.Anthropic()
            msg = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=300,
                messages=[{"role": "user", "content": prompt}],
            )
            return msg.content[0].text
        if self.backend == "openai":
            from openai import OpenAI
            client = OpenAI()
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
            )
            return resp.choices[0].message.content
        # Local deterministic fallback — no API key required
        return self._local_fallback(prompt)

    def _local_fallback(self, prompt: str) -> str:
        return f"[local-planner] processed: {prompt[:120]}"

llm = LLMClient()
print("LLM backend in use:", llm.backend)


LLM backend in use: local


## 4. Agent orchestration — planner, tool selection, memory

The planner breaks a complex query into sub-tasks. Each sub-task is routed to the right tool
(document search vs calculator vs date) automatically, without manual intervention. Intermediate
results are kept in memory and combined by the synthesizer into a single grounded answer.

In [5]:
class ResearchAgent:
    def __init__(self, llm_client, tool_registry):
        self.llm = llm_client
        self.tools = tool_registry
        self.memory = []

    def plan(self, query):
        """Break the query into sub-tasks and assign a tool to each.
        Uses simple pattern rules here; in production this planning step
        is delegated to the LLM (see LLMClient) with the tool list as context."""
        subtasks = [s.strip() for s in re.split(r"\band\b|[?.]", query) if s.strip()]
        plan = []
        for task in subtasks:
            if re.search(r"[0-9].*[%+\-*/]|\b(calculate|percent|%)\b", task, re.I):
                plan.append({"task": task, "tool": "calculator"})
            elif re.search(r"\bdate\b|\btoday\b", task, re.I):
                plan.append({"task": task, "tool": "get_current_date"})
            else:
                plan.append({"task": task, "tool": "search_documents"})
        return plan

    def execute(self, query):
        plan = self.plan(query)
        self.memory.append({"query": query, "plan": plan, "results": []})

        for step in plan:
            tool_name = step["tool"]
            if tool_name == "search_documents":
                result = self.tools.call("search_documents", query=step["task"])
            elif tool_name == "calculator":
                pct_match = re.search(r"(\d+(?:\.\d+)?)\s*%\s*(?:of)?\s*(\d+(?:\.\d+)?)", step["task"])
                if pct_match:
                    pct, base = pct_match.groups()
                    raw_expr = f"({pct}/100)*{base}"
                    result = self.tools.call("calculator", expression=raw_expr)
                else:
                    match = re.search(r"\d[\d\.\s+\-*/()]*\d|\d+", step["task"])
                    result = self.tools.call("calculator", expression=match.group()) if match else None
            elif tool_name == "get_current_date":
                result = self.tools.call("get_current_date")
            else:
                result = None
            self.memory[-1]["results"].append({"tool": tool_name, "task": step["task"], "output": result})

        return self.synthesize(self.memory[-1])

    def synthesize(self, record):
        query_text = record["query"]
        lines = [f"Query: {query_text}", "", "Agent reasoning (tool calls made automatically):"]
        for r in record["results"]:
            tool_name = r["tool"]
            task_name = r["task"]
            output_val = r["output"]
            lines.append(f"  [{tool_name}] task: '{task_name}' -> {output_val}")
        lines.append("")
        lines.append("Synthesized answer:")
        for r in record["results"]:
            if r["tool"] == "search_documents" and r["output"]:
                first_doc = r["output"][0]
                lines.append(f"- {first_doc}")
            elif r["tool"] == "calculator":
                calc_result = r["output"]
                lines.append(f"- Calculation result: {calc_result}")
            elif r["tool"] == "get_current_date":
                today_val = r["output"]
                lines.append(f"- Today's date: {today_val}")
        return "\n".join(lines)

agent = ResearchAgent(llm, tools)


## 5. Run: a multi-step query requiring both RAG retrieval and a tool call

In [6]:
query = "What is the refund policy and what is 15% of 200?"
print(agent.execute(query))


Query: What is the refund policy and what is 15% of 200?

Agent reasoning (tool calls made automatically):
  [search_documents] task: 'What is the refund policy' -> ['Refund Policy: Customers can request a full refund within 30 days of purchase. Refunds are processed to the original payment method within 5-7 business days.', 'Product Warranty: All electronics come with a 1-year manufacturer warranty covering defects. Accidental damage is not covered under the standard warranty.']
  [calculator] task: 'what is 15% of 200' -> 30.0

Synthesized answer:
- Refund Policy: Customers can request a full refund within 30 days of purchase. Refunds are processed to the original payment method within 5-7 business days.
- Calculation result: 30.0


In [7]:
query2 = "How long does express shipping take and what is today\'s date?"
print(agent.execute(query2))


Query: How long does express shipping take and what is today's date?

Agent reasoning (tool calls made automatically):
  [search_documents] task: 'How long does express shipping take' -> ['Shipping Policy: Standard shipping takes 5-8 business days. Express shipping takes 2-3 business days and costs an additional flat fee of $12.']
  [get_current_date] task: 'what is today's date' -> 2026-09-18

Synthesized answer:
- Shipping Policy: Standard shipping takes 5-8 business days. Express shipping takes 2-3 business days and costs an additional flat fee of $12.
- Today's date: 2026-09-18


## 6. Memory: full trace of agent steps across the session

In [8]:
print(json.dumps(agent.memory, indent=2, default=str))


[
  {
    "query": "What is the refund policy and what is 15% of 200?",
    "plan": [
      {
        "task": "What is the refund policy",
        "tool": "search_documents"
      },
      {
        "task": "what is 15% of 200",
        "tool": "calculator"
      }
    ],
    "results": [
      {
        "tool": "search_documents",
        "task": "What is the refund policy",
        "output": [
          "Refund Policy: Customers can request a full refund within 30 days of purchase. Refunds are processed to the original payment method within 5-7 business days.",
          "Product Warranty: All electronics come with a 1-year manufacturer warranty covering defects. Accidental damage is not covered under the standard warranty."
        ]
      },
      {
        "tool": "calculator",
        "task": "what is 15% of 200",
        "output": 30.0
      }
    ]
  },
  {
    "query": "How long does express shipping take and what is today's date?",
    "plan": [
      {
        "task": "How l

## Summary

- **Multi-agent orchestration**: planner decomposes a query into sub-tasks, router assigns each to a tool, synthesizer merges results — no manual intervention between steps
- **RAG**: TF-IDF vector embeddings ground responses in real documents (swap for sentence-transformers / OpenAI embeddings in production)
- **MCP-style tools**: a registry-based tool protocol (name, description, callable) mirroring how MCP exposes tools to an agent at runtime
- **LLM APIs**: pluggable client supports Anthropic/OpenAI when a key is present, with a local fallback so the notebook always runs


## 7. Gradio Agent UI — Purple Theme

Run the agent through a simple interactive interface. The UI shows the final answer, execution trace, and session memory.

In [ ]:
import gradio as gr
from datetime import datetime

PURPLE_CSS = """
:root {
    --purple-main: #7c3aed;
    --purple-dark: #4c1d95;
    --purple-light: #f5f3ff;
}
.gradio-container {
    background: linear-gradient(135deg, #faf5ff 0%, #f3e8ff 48%, #ede9fe 100%) !important;
    font-family: Inter, ui-sans-serif, system-ui, sans-serif;
}
#agent-header {
    background: linear-gradient(135deg, #4c1d95, #7c3aed, #a855f7);
    color: white;
    padding: 24px;
    border-radius: 18px;
    text-align: center;
    box-shadow: 0 10px 30px rgba(76, 29, 149, 0.25);
}
#agent-header h1 { margin: 0; font-size: 30px; }
#agent-header p { margin: 8px 0 0; opacity: 0.9; }
.gr-button-primary {
    background: linear-gradient(90deg, #6d28d9, #9333ea) !important;
    border: none !important;
    color: white !important;
}
textarea, input {
    border-color: #c4b5fd !important;
}
.output-box {
    border: 1px solid #c4b5fd !important;
    border-radius: 14px !important;
}
"""

def run_agent_ui(query, show_trace):
    if not query or not query.strip():
        return "Please enter a research question.", "", json.dumps(agent.memory, indent=2, default=str)

    started = datetime.now().strftime("%H:%M:%S")
    answer = agent.execute(query.strip())
    record = agent.memory[-1] if agent.memory else {}

    trace_lines = [
        f"Run started: {started}",
        f"Detected plan: {record.get('plan', [])}",
        "",
        "Tool execution:"
    ]
    for item in record.get("results", []):
        trace_lines.append(
            f"• {item['tool']} | Task: {item['task']} | Output: {item['output']}"
        )
    trace = "\n".join(trace_lines) if show_trace else "Execution trace hidden. Enable 'Show agent trace'."
    memory = json.dumps(agent.memory, indent=2, default=str)
    return answer, trace, memory

def clear_ui():
    return "", "", "", json.dumps(agent.memory, indent=2, default=str)

with gr.Blocks(css=PURPLE_CSS, theme=gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="purple",
    neutral_hue="slate"
)) as demo:
    gr.HTML("""
    <div id="agent-header">
        <h1>🔮 AI Research Assistant</h1>
        <p>RAG + MCP-style tools + Agent orchestration</p>
    </div>
    """)

    gr.Markdown(
        "Ask a question that may require document retrieval, calculations, or today's date. "
        "The agent automatically plans and executes the required tools."
    )

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Research Query",
                placeholder="Example: What is the refund policy and what is 15% of 200?",
                lines=4
            )
            show_trace = gr.Checkbox(
                label="Show agent trace",
                value=True
            )
            with gr.Row():
                run_btn = gr.Button("▶ Run Agent", variant="primary")
                clear_btn = gr.Button("↺ Clear")

        with gr.Column(scale=3):
            answer_output = gr.Textbox(
                label="✨ Agent Answer",
                lines=10,
                interactive=False,
                elem_classes=["output-box"]
            )

    trace_output = gr.Textbox(
        label="🛠️ Execution Trace",
        lines=10,
        interactive=False,
        elem_classes=["output-box"]
    )
    memory_output = gr.Code(
        label="🧠 Agent Memory",
        language="json",
        lines=12,
        interactive=False
    )

    gr.Examples(
        examples=[
            ["What is the refund policy and what is 15% of 200?", True],
            ["How long does express shipping take and what is today's date?", True],
            ["What is the warranty policy?", True],
            ["Calculate 25% of 800", True],
        ],
        inputs=[query_input, show_trace]
    )

    run_btn.click(
        fn=run_agent_ui,
        inputs=[query_input, show_trace],
        outputs=[answer_output, trace_output, memory_output]
    )
    query_input.submit(
        fn=run_agent_ui,
        inputs=[query_input, show_trace],
        outputs=[answer_output, trace_output, memory_output]
    )
    clear_btn.click(
        fn=clear_ui,
        inputs=[],
        outputs=[query_input, answer_output, trace_output, memory_output]
    )

demo.launch()
